<a href="https://colab.research.google.com/github/SaraAlinejad/vae_test_1/blob/main/Alinejad_assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 4 - Deadline 3/6/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Add your name here

**AI usage statement:**
Here you should give a statement about any usage of AI tools to assist you with the coding.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Perform regression using random forest regression model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: use the default values
- Maximum number of features for each tree: use the default values

You should use the log10 transformed passive permeability values as target values.

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance. You should select the metrics you wish to use.

Based on your analysis, which feature set gives the best results for the regression? Does these results fit with the results you obtained in Assignment 3?

#### D) - Optional for 2 points
For the best performing feature set from A), repeat the analysis using the measured passive permeability values in nm/s (i.e., the non log10 transformed values). Do you observe any measurable difference in using the direct values or the log10 transformed values?


In [ ]:
%%bash
# Download the main model data and the molecular structure data
BASE_URL="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/main"
rm -f model_data.csv mol_data.csv

wget ${BASE_URL}/outputs/ml_models/model_data.csv &> /dev/null
wget ${BASE_URL}/data/mol_data.csv &> /dev/null

ls -l *.csv

In [ ]:
!pip install rdkit mols2grid lux-api -q # lux is useful visualization package for dataframes
import warnings
warnings.filterwarnings("ignore")
import math
import pandas as pd
import numpy as np
import lux  #on top of the dataframe, there will be a tab; Toggle Pandas/lux, click on it and all usufull plots can be seen
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import seaborn  as sns
import mols2grid
from rdkit import Chem
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate, ShuffleSplit, StratifiedKFold
from sklearn.metrics import make_scorer, accuracy_score, precision_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
#  A clean, style that looks better

PALETTE_LOW  = "#E05C5C"   # warm red  : low permeability
PALETTE_HIGH = "#4A90D9"   # steel blue : high permeability
PALETTE      = {"Low": PALETTE_LOW, "High": PALETTE_HIGH}

plt.rcParams.update({
    "figure.facecolor"     : "white",
    "axes.facecolor"       : "#F8F9FA",
    "axes.edgecolor"       : "#CCCCCC",
    "axes.grid"            : True,
    "grid.color"           : "white",
    "grid.linewidth"       : 1.2,
    "axes.spines.top"      : False,
    "axes.spines.right"    : False,
    "axes.labelsize"       : 12,
    "axes.titlesize"       : 13,
    "axes.titleweight"     : "bold",
    "xtick.labelsize"      : 10,
    "ytick.labelsize"      : 10,
    "legend.frameon"       : False,
    "legend.fontsize"      : 10,
    "font.family"          : "DejaVu Sans",
    "figure.dpi"           : 130,
})

In [ ]:
df = pd.read_csv("model_data.csv")
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
TARGET     = "P_appLog"   # log10-transformed passive permeability

FEATURES_2D = [
    "Molecular Weight (MW)",
    "CharVol (characteristic volume)",
    "Flexibility (number of rotatable bonds / number of bonds)",
    "Number of Heavy Atoms (HA)",
    "RingAtoms", "Halogens", "HeteroAtoms",
    "RotBonds (NRotB)", "AllBonds", "RingCount", "NumStereo",
    "Fraction of sp3 Carbon Atoms (FSP3)",
    "Hydrogen Bond Donors (HBD)",
    "Hydrogen Bond Acceptors (HBA)",
    "cLogD^7.4",
    "Topological polar surface area (TPSA)",
    "Total non-polar surface area (TNSA)",
]

FEATURES_3D = [
    "Ensemble_Average_PSA_Chloroform_ANI",
    "Ensemble_Average_Num_IMHB_Chloroform_ANI",
    "Ensemble_Average_RadiusOfGyration_Chloroform_ANI",
]

# Verifying all expected columns
missing = [c for c in FEATURES_2D + FEATURES_3D + [TARGET] if c not in df.columns]
if missing:
    print(" Missing columns:", missing)
else:
    print("All feature columns present.")

In [ ]:
#  Recover raw P_app from the log-transformed column
# The CSV stores log10(P_app); back-transform to nm/s for the threshold
if "P_app" not in df.columns:
    df["P_app"] = 10 ** df[TARGET]

# Compute median and assign labels
THRESHOLD = df["P_app"].median()   # nm/s

# Compounds strictly ABOVE the median :High; otherwise : Low
# Using strict inequality on an even-N dataset guarantees 16 / 16
df["Class"] = np.where(df["P_app"] > THRESHOLD, "High", "Low")

#  Sanity check
counts = df["Class"].value_counts()
print(f"Median P_app threshold : {THRESHOLD:.4f} nm/s")
print(f"Class counts           : {dict(counts)}")
assert counts["High"] == 16 and counts["Low"] == 16, "Split is not balanced!"
print(" Balanced split confirmed: 16 Low · 16 High")

In [ ]:
# Ranked bar chart — to show the split
plot_df = df.sort_values("P_app").reset_index(drop=True)
plot_df["rank"] = range(1, 33)

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = [PALETTE[c] for c in plot_df["Class"]]
ax.bar(plot_df["rank"], plot_df["P_app"],
       color=bar_colors, edgecolor="white", linewidth=0.6, width=0.85)

ax.axhline(THRESHOLD, color="#1a1a2e", lw=2, ls="--", zorder=5)
ax.text(33, THRESHOLD + 0.4,
        f" Median = {THRESHOLD:.1f} nm/s",
        va="bottom", ha="left", fontsize=9, color="#1a1a2e", style="italic")

ax.text(2, THRESHOLD * 0.38, "Low (n = 16)", ha="left",
        fontsize=11, color=PALETTE["Low"], fontweight="bold",
        zorder=20)

ax.text(18, THRESHOLD * 1.55, "High (n = 16)", ha="left",
        fontsize=11, color=PALETTE["High"], fontweight="bold",
        zorder=20)

ax.legend(
    handles=[mpatches.Patch(facecolor=PALETTE[c], label=c) for c in ["Low", "High"]],
    title="Class", loc="upper left"
)
ax.set_xlabel("Compound rank (ascending $P_{app}$)")
ax.set_ylabel(r"$P_{app}$ (nm s$^{-1}$)")
ax.set_title("Median Split into Low / High Permeability Classes\n"
             "32 Heterobifunctional Degraders")
ax.set_xlim(0.3, 32.7)
plt.tight_layout()
plt.show()

In [ ]:
feat_df = pd.read_csv("model_data.csv")   # descriptors + P_appLog
mol_df  = pd.read_csv("mol_data.csv")     # Smiles + P_app

# ── Check which P_app values are duplicated ───────────────────────────────────
print("Duplicate P_app values in mol_data.csv:")
print(mol_df[mol_df["P_app"].duplicated(keep=False)][["Compound","P_app"]])

In [ ]:
#Sort both files by P_appLog , same ordering

mol_df["_log_key"] = np.log10(mol_df["P_app"].round(1))
mol_sorted  = mol_df.sort_values("_log_key").reset_index(drop=True)
feat_sorted = feat_df.sort_values("P_appLog").reset_index(drop=True)

#  Positional concat
merged = pd.concat([mol_sorted, feat_sorted], axis=1)
merged = merged.loc[:, ~merged.columns.duplicated()]
merged = merged.drop(columns=["_log_key"])

print(f"Merged shape : {merged.shape}")   # (32, ...)

#  Verify
diff = np.abs(np.log10(merged["P_app"].round(1)) - merged["P_appLog"])
print(f"Max diff     : {diff.max():.8f}")

#  Spot-check
print(merged[merged["P_app"].round(1) == 3.2][["Compound", "P_app", "P_appLog"]])

In [ ]:
#  Median split
THRESHOLD = merged["P_app"].median()
merged["Class"]        = np.where(merged["P_app"] > THRESHOLD, "High", "Low")
merged["P_app (nm/s)"] = merged["P_app"].round(1)

print(f"Threshold : {THRESHOLD:.4f} nm/s")
print(f"Classes   : {dict(merged['Class'].value_counts())}")   #  16 / 16

In [ ]:
# ── Feature matrices
X_2D  = merged[FEATURES_2D].values
X_3D  = merged[FEATURES_3D].values
X_all = np.hstack([X_2D, X_3D])

# ── Target: High = 1 (positive class), Low = 0
y = (merged["Class"] == "High").astype(int).values

print(f"X_2D  shape : {X_2D.shape}   : max_features = {math.ceil(0.5 * X_2D.shape[1])}")
print(f"X_3D  shape : {X_3D.shape}   : max_features = {math.ceil(0.5 * X_3D.shape[1])}")
print(f"X_all shape : {X_all.shape}  : max_features = {math.ceil(0.5 * X_all.shape[1])}")
print(f"\nTarget: {dict(zip(*np.unique(y, return_counts=True)))}  (1=High, 0=Low)")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (RepeatedKFold, LeaveOneOut,
                                     cross_validate, cross_val_predict)
from sklearn.metrics import (make_scorer, mean_absolute_error,
                             mean_squared_error, r2_score, fbeta_score)
import warnings; warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestRegressor

from sklearn.datasets import make_regression

In [ ]:
y_log = df['P_appLog'].values                # log10(P_app)

 y_raw =y_log

In [ ]:
RF = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
CV = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

def rmse(y, yp):   return np.sqrt(mean_squared_error(y, yp))
def spear(y, yp):  return stats.spearmanr(y, yp).correlation

In [ ]:
scorers = {
    'R2'      : make_scorer(r2_score),
    'MAE'     : make_scorer(mean_absolute_error),
    'RMSE'    : make_scorer(rmse),
    'Spearman': make_scorer(spear),
}

In [ ]:
def run_cv(X,y, label):
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    raw = cross_validate(model, X, y, cv=CV, scoring=scorers, return_train_score=False)
    res = {m: {'mean': raw[f'test_{m}'].mean(),'std' : raw[f'test_{m}'].std(), 'all' : raw[f'test_{m}']}
           for m in scorers}
    print(f"\n{'─'*50}\n  {label}")
    for m, lbl in zip(['R2','MAE','RMSE','Spearman'],
                      ['R²  ','MAE ','RMSE','p   ']):
        print(f"  {lbl}: {res[m]['mean']:+.4f} ± {res[m]['std']:.4f}")
    return res

In [ ]:
X2d   = df[FEATURES_2D].values
X3d   = df[FEATURES_3D].values
features_combined = FEATURES_2D + FEATURES_3D
Xcomb = df[features_combined].values

r_2d   = run_cv(X2d, y_log, "2D RDKit  (17 feat) — log10 target")
r_3d   = run_cv(X3d, y_log, "3D Ensemb ( 3 feat) — log10 target")
r_comb = run_cv(Xcomb, y_log, "2D+3D    (20 feat) — log10 target")

In [ ]:
metrics   = ["R2", "MAE", "RMSE"]
fs_names  = list(results_knn.keys())
fs_colors = ["#5B8DB8", "#E8A838", "#6CBB6C"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, metric in zip(axes, metrics):
    means = [r_comb[fs][metric].mean() for fs in fs_names]
    stds  = [r_comb[fs][metric].std()  for fs in fs_names]

    bars = ax.bar(fs_names, means, color=fs_colors,
                  edgecolor="white", linewidth=0.8, width=0.55,
                  yerr=stds, capsize=6)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02,
                f"{m:.2f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold")

    ax.set_ylim(0, 1.18)
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("Feature set")

fig.suptitle(" 5-fold Stratified CV | Error = ±1 SD",
             fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
scorers_extra = {
    'fbeta'      : make_scorer(fbeta_score, beta=2),
}

In [ ]:
make_scorer(fbeta_score, response_method='predict', beta=2)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
grid = GridSearchCV(LinearSVC(), param_grid={'C': [1, 10]},
                    scoring=ftwo_scorer)